In [1]:
# ─────────────────────────────────────────────────────────────
# 02 FEATURES — LANL Authentication Dataset
# Builds genuinely correct features from real LANL fields
# (success_failure, logon_type, timestamps) — no derived bugs
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

df = pd.read_csv('parsed_logs.csv')
df = df.sort_values('time').reset_index(drop=True)

print(f"Loaded {len(df):,} events")
print(f"Attack ratio: {df['is_attack'].mean()*100:.2f}%")
print(f"\nColumns: {df.columns.tolist()}")

Loaded 68,221 events
Attack ratio: 14.98%

Columns: ['time', 'source_user', 'destination_user', 'source_computer', 'destination_computer', 'authentication_type', 'logon_type', 'authentication_orientation', 'success_failure', 'is_attack']


In [2]:
# ── Feature 1: is_failed_login ───────────────────────────────
# Directly from the genuine success_failure column — no EventID
# guessing, no inverted conditions. This is the real signal that
# was structurally impossible to compute correctly in the OTRF
# dataset (zero EventID 4625 events existed there).

df['is_failed_login'] = (df['success_failure'] == 'Fail').astype(int)

print("is_failed_login distribution:")
print(df['is_failed_login'].value_counts())
print(f"\nFailure rate: {df['is_failed_login'].mean()*100:.3f}%")

# Sanity check: does this align with attack accounts having
# elevated failure rates compared to normal accounts?
print(f"\nFailure rate — attack accounts: {df[df['is_attack']==1]['is_failed_login'].mean()*100:.3f}%")
print(f"Failure rate — normal accounts: {df[df['is_attack']==0]['is_failed_login'].mean()*100:.3f}%")

is_failed_login distribution:
is_failed_login
0    67630
1      591
Name: count, dtype: int64

Failure rate: 0.866%

Failure rate — attack accounts: 0.205%
Failure rate — normal accounts: 0.983%


In [3]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['is_attack'], df['is_failed_login'])
print("Contingency table:")
print(contingency)

chi2, p_value, dof, expected = chi2_contingency(contingency)
print(f"\nChi-square statistic: {chi2:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Significant at p<0.05: {p_value < 0.05}")

Contingency table:
is_failed_login      0    1
is_attack                  
0                57430  570
1                10200   21

Chi-square statistic: 60.2331
p-value: 0.000000
Significant at p<0.05: True


In [4]:
# ── Feature 2: logins_per_hour ───────────────────────────────
# Genuine rolling 60-minute window count per source_user,
# computed from real Unix timestamps. This directly fixes the
# OTRF bug where "logins_per_hour" was actually a static total
# event count per user (confirmed via 100% match test earlier).

df = df.sort_values(['source_user', 'time']).reset_index(drop=True)

def rolling_login_count(group, window_seconds=3600):
    times = group['time'].values
    counts = np.zeros(len(times), dtype=int)
    start_idx = 0
    for i in range(len(times)):
        while times[i] - times[start_idx] > window_seconds:
            start_idx += 1
        counts[i] = i - start_idx + 1
    return counts

print("Computing genuine rolling 60-minute login counts per user...")
print("(this may take a moment for 68,221 events)\n")

df['logins_per_hour'] = (
    df.groupby('source_user', group_keys=False)
    .apply(lambda g: pd.Series(rolling_login_count(g), index=g.index))
)

print("logins_per_hour distribution:")
print(df['logins_per_hour'].describe())

print(f"\nMean logins_per_hour — attack accounts: {df[df['is_attack']==1]['logins_per_hour'].mean():.2f}")
print(f"Mean logins_per_hour — normal accounts: {df[df['is_attack']==0]['logins_per_hour'].mean():.2f}")

# Confirm it's NOT just a static count per user (the OTRF bug check)
print(f"\nIs this constant per user? (should be False/varied this time)")
variance_check = df.groupby('source_user')['logins_per_hour'].nunique()
print(f"Users with more than 1 unique value: {(variance_check > 1).sum():,} out of {len(variance_check):,}")

Computing genuine rolling 60-minute login counts per user...
(this may take a moment for 68,221 events)

logins_per_hour distribution:
count    68221.000000
mean        10.473945
std         25.797109
min          1.000000
25%          1.000000
50%          2.000000
75%          6.000000
max        225.000000
Name: logins_per_hour, dtype: float64

Mean logins_per_hour — attack accounts: 6.86
Mean logins_per_hour — normal accounts: 11.11

Is this constant per user? (should be False/varied this time)
Users with more than 1 unique value: 6,848 out of 14,309


In [5]:
# Check why ~7,461 users show only 1 unique value — 
# is this a legitimate single-event user, or a bug?
single_value_users = variance_check[variance_check == 1].index
single_value_sample = df[df['source_user'].isin(single_value_users[:5])]
print("Sample of users with only 1 unique logins_per_hour value:")
print(single_value_sample[['source_user', 'time', 'logins_per_hour']].sort_values('source_user'))

print(f"\nHow many events do these 'single value' users have on average?")
events_per_single_value_user = df[df['source_user'].isin(single_value_users)].groupby('source_user').size()
print(events_per_single_value_user.describe())

Sample of users with only 1 unique logins_per_hour value:
               source_user     time  logins_per_hour
0   ANONYMOUS LOGON@C10561  1066215                1
45   ANONYMOUS LOGON@C1208  1046413                1
46   ANONYMOUS LOGON@C1208  1054767                1
47   ANONYMOUS LOGON@C1208  1068012                1
48  ANONYMOUS LOGON@C12166  1066449                1
49  ANONYMOUS LOGON@C12214  1064362                1
50  ANONYMOUS LOGON@C12349  1066167                1

How many events do these 'single value' users have on average?
count    7461.000000
mean        1.627262
std         0.860078
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         6.000000
dtype: float64


In [6]:
from scipy.stats import mannwhitneyu

attack_velocity = df[df['is_attack']==1]['logins_per_hour']
normal_velocity = df[df['is_attack']==0]['logins_per_hour']

stat, p_value = mannwhitneyu(attack_velocity, normal_velocity, alternative='two-sided')
print(f"Mann-Whitney U statistic: {stat:.2f}")
print(f"p-value: {p_value:.6f}")
print(f"Significant at p<0.05: {p_value < 0.05}")

Mann-Whitney U statistic: 410810816.00
p-value: 0.000000
Significant at p<0.05: True


In [7]:
# ── Feature 3: unique_destination_computers ──────────────────
# Distinct destination computers per source_user across the
# entire dataset window — direct lateral movement indicator

dest_computer_counts = df.groupby('source_user')['destination_computer'].transform('nunique')
df['unique_destination_computers'] = dest_computer_counts

print("unique_destination_computers distribution:")
print(df['unique_destination_computers'].describe())

print(f"\nMean — attack accounts: {df[df['is_attack']==1]['unique_destination_computers'].mean():.2f}")
print(f"Mean — normal accounts: {df[df['is_attack']==0]['unique_destination_computers'].mean():.2f}")

from scipy.stats import mannwhitneyu
attack_dest = df[df['is_attack']==1]['unique_destination_computers']
normal_dest = df[df['is_attack']==0]['unique_destination_computers']
stat, p_value = mannwhitneyu(attack_dest, normal_dest, alternative='two-sided')
print(f"\nMann-Whitney U statistic: {stat:.2f}")
print(f"p-value: {p_value:.6f}")
print(f"Significant at p<0.05: {p_value < 0.05}")

unique_destination_computers distribution:
count    68221.000000
mean         6.565515
std          8.051779
min          1.000000
25%          2.000000
50%          3.000000
75%          7.000000
max         66.000000
Name: unique_destination_computers, dtype: float64

Mean — attack accounts: 20.24
Mean — normal accounts: 4.16

Mann-Whitney U statistic: 568542227.50
p-value: 0.000000
Significant at p<0.05: True


In [8]:
# ── Feature 4: is_privileged_target ──────────────────────────
# Identifies "server-like" destination computers using fan-in:
# computers authenticated to by an unusually large number of
# distinct source users are likely servers/domain controllers,
# not personal workstations

dest_fanin = df.groupby('destination_computer')['source_user'].transform('nunique')
df['destination_fanin'] = dest_fanin

threshold = df['destination_fanin'].quantile(0.95)
df['is_privileged_target'] = (df['destination_fanin'] >= threshold).astype(int)

print(f"Fan-in threshold (95th percentile): {threshold:.0f} distinct users")
print(f"\nis_privileged_target distribution:")
print(df['is_privileged_target'].value_counts())

print(f"\nMean is_privileged_target — attack accounts: {df[df['is_attack']==1]['is_privileged_target'].mean()*100:.2f}%")
print(f"Mean is_privileged_target — normal accounts: {df[df['is_attack']==0]['is_privileged_target'].mean()*100:.2f}%")

from scipy.stats import chi2_contingency
contingency = pd.crosstab(df['is_attack'], df['is_privileged_target'])
print(f"\nContingency table:")
print(contingency)
chi2, p_value, dof, expected = chi2_contingency(contingency)
print(f"\nChi-square statistic: {chi2:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Significant at p<0.05: {p_value < 0.05}")

Fan-in threshold (95th percentile): 3441 distinct users

is_privileged_target distribution:
is_privileged_target
0    59946
1     8275
Name: count, dtype: int64

Mean is_privileged_target — attack accounts: 5.48%
Mean is_privileged_target — normal accounts: 13.30%

Contingency table:
is_privileged_target      0     1
is_attack                        
0                     50285  7715
1                      9661   560

Chi-square statistic: 498.1929
p-value: 0.000000
Significant at p<0.05: True


In [9]:
# ── Feature 5: Encode categorical fields ─────────────────────
from sklearn.preprocessing import LabelEncoder

le_user = LabelEncoder()
le_dest_computer = LabelEncoder()
le_logon_type = LabelEncoder()
le_auth_type = LabelEncoder()
le_orientation = LabelEncoder()

df['source_user_encoded'] = le_user.fit_transform(df['source_user'])
df['destination_computer_encoded'] = le_dest_computer.fit_transform(df['destination_computer'])
df['logon_type_encoded'] = le_logon_type.fit_transform(df['logon_type'].fillna('Unknown'))
df['authentication_type_encoded'] = le_auth_type.fit_transform(df['authentication_type'].fillna('Unknown'))
df['authentication_orientation_encoded'] = le_orientation.fit_transform(df['authentication_orientation'])

print("Categorical encoding complete:")
print(f"  Distinct source users: {df['source_user_encoded'].nunique():,}")
print(f"  Distinct destination computers: {df['destination_computer_encoded'].nunique():,}")
print(f"  Distinct logon types: {df['logon_type_encoded'].nunique()}")
print(f"  Distinct authentication types: {df['authentication_type_encoded'].nunique()}")
print(f"  Distinct authentication orientations: {df['authentication_orientation_encoded'].nunique()}")

Categorical encoding complete:
  Distinct source users: 14,309
  Distinct destination computers: 3,520
  Distinct logon types: 10
  Distinct authentication types: 8
  Distinct authentication orientations: 7


In [10]:
# ── Assemble final feature matrix ────────────────────────────
feature_columns = [
    'is_failed_login',
    'logins_per_hour',
    'unique_destination_computers',
    'is_privileged_target',
    'destination_fanin',
    'source_user_encoded',
    'destination_computer_encoded',
    'logon_type_encoded',
    'authentication_type_encoded',
    'authentication_orientation_encoded',
]

features_df = df[feature_columns].copy()

print(f"Final feature matrix shape: {features_df.shape}")
print(f"\nFeatures included ({len(feature_columns)} total):")
for col in feature_columns:
    print(f"  - {col}")

print(f"\nMissing values check:")
print(features_df.isnull().sum())

print(f"\nFeature summary statistics:")
print(features_df.describe())

Final feature matrix shape: (68221, 10)

Features included (10 total):
  - is_failed_login
  - logins_per_hour
  - unique_destination_computers
  - is_privileged_target
  - destination_fanin
  - source_user_encoded
  - destination_computer_encoded
  - logon_type_encoded
  - authentication_type_encoded
  - authentication_orientation_encoded

Missing values check:
is_failed_login                       0
logins_per_hour                       0
unique_destination_computers          0
is_privileged_target                  0
destination_fanin                     0
source_user_encoded                   0
destination_computer_encoded          0
logon_type_encoded                    0
authentication_type_encoded           0
authentication_orientation_encoded    0
dtype: int64

Feature summary statistics:
       is_failed_login  logins_per_hour  unique_destination_computers  \
count     68221.000000     68221.000000                  68221.000000   
mean          0.008663        10.473945        

In [11]:
# ── Save outputs ──────────────────────────────────────────────
features_df.to_csv('features.csv', index=False)

# Save full df (with labels + all engineered columns) for
# evaluation and SHAP stages later
df.to_csv('df_with_features.csv', index=False)

print(f"Saved features.csv — shape: {features_df.shape}")
print(f"Saved df_with_features.csv — shape: {df.shape}")

import os
print(f"\nFile sizes:")
print(f"  features.csv: {os.path.getsize('features.csv') / (1024**2):.2f} MB")
print(f"  df_with_features.csv: {os.path.getsize('df_with_features.csv') / (1024**2):.2f} MB")

Saved features.csv — shape: (68221, 10)
Saved df_with_features.csv — shape: (68221, 20)

File sizes:
  features.csv: 1.93 MB
  df_with_features.csv: 6.52 MB


## Objective: Feature Engineering (LANL Dataset)

**Status: COMPLETE — All features statistically validated against ground truth**

| Feature | Attack Mean | Normal Mean | Test | p-value | Significant |
|---|---|---|---|---|---|
| is_failed_login | 0.205% | 0.983% | Chi-square | <0.000001 | Yes (inverse) |
| logins_per_hour | 6.86 | 11.11 | Mann-Whitney U | <0.000001 | Yes (inverse) |
| unique_destination_computers | 20.24 | 4.16 | Mann-Whitney U | <0.000001 | Yes (5x higher) |
| is_privileged_target | 5.48% | 13.30% | Chi-square | <0.000001 | Yes (inverse) |

**Key finding**: This LANL attack scenario represents low-and-slow credential reuse — attackers show LOWER failure rates, LOWER login velocity, and LOWER targeting of high-traffic infrastructure than normal users, but 5x HIGHER lateral movement breadth. This is the primary attack signature in this dataset and directly informs SIEM rule design in the next notebook.

Output: features.csv (68,221 × 10, zero missing values), df_with_features.csv (68,221 × 20)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('parsed_logs.csv')
print(f"Raw events: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

# LANL time is seconds from epoch=1
# Convert to day number by dividing by 86400
df['day'] = (df['time'] // 86400).astype(int)

print(f"\nTime range in seconds: {df['time'].min():,} to {df['time'].max():,}")
print(f"Day range: {df['day'].min()} to {df['day'].max()}")
print(f"Distinct days in dataset: {df['day'].nunique()}")
print(f"Distinct source_users: {df['source_user'].nunique():,}")
print(f"\nDistinct (user, day) combinations: {df.groupby(['source_user','day']).ngroups:,}")

# Check attack accounts
ATTACK_ACCOUNTS = set(df[df['is_attack']==1]['source_user'].unique())
print(f"\nDistinct attack accounts: {len(ATTACK_ACCOUNTS)}")
print(f"Sample attack accounts: {list(ATTACK_ACCOUNTS)[:5]}")

Raw events: 68,221
Columns: ['time', 'source_user', 'destination_user', 'source_computer', 'destination_computer', 'authentication_type', 'logon_type', 'authentication_orientation', 'success_failure', 'is_attack']

Time range in seconds: 1,036,800 to 1,295,848
Day range: 12 to 14
Distinct days in dataset: 3
Distinct source_users: 14,309

Distinct (user, day) combinations: 14,416

Distinct attack accounts: 60
Sample attack accounts: ['U342@DOM1', 'U825@DOM1', 'U1653@DOM1', 'U1581@DOM1', 'U5254@DOM1']


In [2]:
# ── REBUILT 02_features.ipynb ────────────────────────────────
# User-day aggregation following Microsoft Sentinel's published
# best practice for multi-feature Isolation Forest on authentication logs
# Reference: Patil, A. (2023). Microsoft Sentinel Blog.

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('parsed_logs.csv')
df['day'] = (df['time'] // 86400).astype(int)

# ── Ground truth — attack account labels ─────────────────────
ATTACK_ACCOUNTS = set(df[df['is_attack']==1]['source_user'].unique())
print(f"Attack accounts: {len(ATTACK_ACCOUNTS)}")

# ── Encode categorical fields for aggregation ─────────────────
df['is_failed'] = (df['success_failure'] == 'Fail').astype(int)
df['is_success'] = (df['success_failure'] == 'Success').astype(int)
df['is_batch_or_rdp'] = df['logon_type'].isin(['Batch','RemoteInteractive']).astype(int)
df['is_network_logon'] = (df['logon_type'] == 'Network').astype(int)
df['is_kerberos'] = (df['authentication_type'] == 'Kerberos').astype(int)
df['is_ntlm'] = (df['authentication_type'] == 'NTLM').astype(int)

# ── Aggregate to user-day level ───────────────────────────────
print("\nAggregating to user-day level...")

agg = df.groupby(['source_user', 'day']).agg(

    # Volume features
    total_logons          = ('time', 'count'),
    failed_logons         = ('is_failed', 'sum'),
    successful_logons     = ('is_success', 'sum'),

    # Rate features
    failure_rate          = ('is_failed', 'mean'),
    success_rate          = ('is_success', 'mean'),

    # Lateral movement
    unique_destinations   = ('destination_computer', 'nunique'),
    unique_source_computers = ('source_computer', 'nunique'),

    # Logon type diversity
    unique_logon_types    = ('logon_type', 'nunique'),
    batch_or_rdp_logons   = ('is_batch_or_rdp', 'sum'),
    network_logons        = ('is_network_logon', 'sum'),

    # Authentication protocol diversity
    unique_auth_types     = ('authentication_type', 'nunique'),
    kerberos_logons       = ('is_kerberos', 'sum'),
    ntlm_logons           = ('is_ntlm', 'sum'),

    # Temporal spread
    active_hours          = ('time', lambda x: (x.max()-x.min())/3600),

).reset_index()

# ── Ground truth label ────────────────────────────────────────
agg['is_attack'] = agg['source_user'].isin(ATTACK_ACCOUNTS).astype(int)

print(f"User-day rows: {len(agg):,}")
print(f"Attack user-days: {agg['is_attack'].sum():,} ({agg['is_attack'].mean()*100:.2f}%)")
print(f"Normal user-days: {(agg['is_attack']==0).sum():,}")
print(f"\nFeatures engineered: {len([c for c in agg.columns if c not in ['source_user','day','is_attack']])}")
print(f"\nSample aggregated rows:")
print(agg.head())

Attack accounts: 60

Aggregating to user-day level...
User-day rows: 14,416
Attack user-days: 167 (1.16%)
Normal user-days: 14,249

Features engineered: 14

Sample aggregated rows:
              source_user  day  total_logons  failed_logons  \
0  ANONYMOUS LOGON@C10561   12             1              0   
1   ANONYMOUS LOGON@C1065   12            44              0   
2   ANONYMOUS LOGON@C1208   12             3              0   
3  ANONYMOUS LOGON@C12166   12             1              0   
4  ANONYMOUS LOGON@C12214   12             1              0   

   successful_logons  failure_rate  success_rate  unique_destinations  \
0                  1           0.0           1.0                    1   
1                 44           0.0           1.0                    1   
2                  3           0.0           1.0                    1   
3                  1           0.0           1.0                    1   
4                  1           0.0           1.0                    1   

 

In [3]:
from scipy.stats import mannwhitneyu, chi2_contingency

print("Statistical validation of user-day features")
print("="*65)

continuous_features = ['total_logons', 'failed_logons', 'successful_logons',
                        'failure_rate', 'unique_destinations',
                        'unique_source_computers', 'unique_logon_types',
                        'batch_or_rdp_logons', 'network_logons',
                        'unique_auth_types', 'kerberos_logons',
                        'ntlm_logons', 'active_hours']

attack = agg[agg['is_attack']==1]
normal = agg[agg['is_attack']==0]

results = []
for feat in continuous_features:
    atk_mean = attack[feat].mean()
    nrm_mean = normal[feat].mean()
    stat, p = mannwhitneyu(attack[feat], normal[feat], alternative='two-sided')
    direction = 'HIGHER' if atk_mean > nrm_mean else 'LOWER'
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    print(f"{feat:<28} attack={atk_mean:>8.3f} normal={nrm_mean:>8.3f} "
          f"({direction}) p={p:.4f} {sig}")
    results.append({'feature': feat, 'attack_mean': atk_mean,
                    'normal_mean': nrm_mean, 'p_value': p,
                    'direction': direction, 'significant': p < 0.05})

sig_count = sum(r['significant'] for r in results)
print(f"\n{sig_count}/{len(continuous_features)} features statistically significant at p<0.05")

Statistical validation of user-day features
total_logons                 attack=  61.204 normal=   4.070 (HIGHER) p=0.0000 ***
failed_logons                attack=   0.126 normal=   0.040 (HIGHER) p=0.0000 ***
successful_logons            attack=  61.078 normal=   4.030 (HIGHER) p=0.0000 ***
failure_rate                 attack=   0.007 normal=   0.007 (HIGHER) p=0.0000 ***
unique_destinations          attack=  11.922 normal=   2.067 (HIGHER) p=0.0000 ***
unique_source_computers      attack=  10.174 normal=   1.944 (HIGHER) p=0.0000 ***
unique_logon_types           attack=   2.407 normal=   1.282 (HIGHER) p=0.0000 ***
batch_or_rdp_logons          attack=   0.431 normal=   0.007 (HIGHER) p=0.0000 ***
network_logons               attack=  43.389 normal=   3.384 (HIGHER) p=0.0000 ***
unique_auth_types            attack=   2.497 normal=   1.555 (HIGHER) p=0.0000 ***
kerberos_logons              attack=  17.395 normal=   1.441 (HIGHER) p=0.0000 ***
ntlm_logons                  attack=   2.91

In [4]:
# ── Save the aggregated dataset ───────────────────────────────
agg.to_csv('df_with_features.csv', index=False)

# ── Build the feature matrix for Isolation Forest ─────────────
feature_columns = [
    'total_logons', 'failed_logons', 'failure_rate',
    'unique_destinations', 'unique_source_computers',
    'unique_logon_types', 'batch_or_rdp_logons',
    'network_logons', 'unique_auth_types',
    'kerberos_logons', 'ntlm_logons', 'active_hours'
]

# Drop success_rate and successful_logons — redundant with
# failure_rate and total_logons (perfectly correlated),
# would add noise without new information
X = agg[feature_columns].copy()
X = X.fillna(0)  # any NaN from division by zero → 0

print(f"Feature matrix shape: {X.shape}")
print(f"\nMissing values check:")
print(X.isnull().sum())
print(f"\nFeature statistics:")
print(X.describe())

X.to_csv('features.csv', index=False)
print(f"\nSaved features.csv — {X.shape[0]:,} user-days × {X.shape[1]} features")
print(f"Saved df_with_features.csv")

Feature matrix shape: (14416, 12)

Missing values check:
total_logons               0
failed_logons              0
failure_rate               0
unique_destinations        0
unique_source_computers    0
unique_logon_types         0
batch_or_rdp_logons        0
network_logons             0
unique_auth_types          0
kerberos_logons            0
ntlm_logons                0
active_hours               0
dtype: int64

Feature statistics:
       total_logons  failed_logons  failure_rate  unique_destinations  \
count  14416.000000   14416.000000  14416.000000         14416.000000   
mean       4.732311       0.040996      0.006739             2.180702   
std       19.795696       2.438118      0.079311             1.795517   
min        1.000000       0.000000      0.000000             1.000000   
25%        1.000000       0.000000      0.000000             1.000000   
50%        2.000000       0.000000      0.000000             2.000000   
75%        4.000000       0.000000      0.000000  

In [6]:
results = {}
for hours, seconds in [('1 hour', 3600), ('4 hours', 14400),
                        ('8 hours', 28800), ('12 hours', 43200),
                        ('1 day', 86400)]:
    df['window'] = (df['time'] // seconds).astype(int)
    events_per_window = df.groupby(['source_user','window']).size()
    attack_windows = df[df['source_user'].isin(attack_accounts)].groupby(
        ['source_user','window']).size()
    results[hours] = {
        'total_windows': len(events_per_window),
        'median_events': events_per_window.median(),
        'pct_single': (events_per_window==1).mean()*100,
        'pct_5plus': (events_per_window>=5).mean()*100,
        'attack_windows': len(attack_windows),
        'attack_median': attack_windows.median(),
    }

print(f"{'Window':>10} {'Total':>8} {'Median':>8} {'%Single':>9} "
      f"{'%5+events':>11} {'Atk Windows':>13} {'Atk Median':>11}")
print("-"*75)
for hours, r in results.items():
    print(f"{hours:>10} {r['total_windows']:>8,} {r['median_events']:>8.1f} "
          f"{r['pct_single']:>8.1f}% {r['pct_5plus']:>10.1f}% "
          f"{r['attack_windows']:>13,} {r['attack_median']:>11.1f}")

    Window    Total   Median   %Single   %5+events   Atk Windows  Atk Median
---------------------------------------------------------------------------
    1 hour   35,453      1.0     69.3%        5.5%         2,520         3.0
   4 hours   23,903      1.0     51.2%        8.7%           843        10.0
   8 hours   17,930      2.0     41.1%       14.7%           460        19.0
  12 hours   14,568      2.0     29.4%       20.1%           319        31.0
     1 day   14,416      2.0     29.7%       19.3%           167        63.0
